In [ ]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

class StopExecution(Exception):
    def _render_traceback_(self):
        return []



In [ ]:
import traceback
import warnings
import sys

def warn_with_traceback(message, category, filename, lineno, file=None, line=None):

    log = file if hasattr(file,'write') else sys.stderr
    traceback.print_stack(file=log)
    log.write(warnings.formatwarning(message, category, filename, lineno, line))

#warnings.showwarning = None

In [ ]:
import sys
sys.path.append("..")
import pickle as pkl
import hist
from analyzer.datasets import SampleManager
from analyzer.core import AnalysisResult
import math
import torch
import gpytorch
from torch.masked import masked_tensor, as_masked_tensor
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from analyzer.plotting.plots_1d import drawAs1DHist
from analyzer.plotting.plots_2d import drawAs2DHist, addTitles2D
from analyzer.plotting.plottables import PlotObject
from analyzer.plotting.mplstyles import loadStyles
import fitting.regression as regression
import fitting.plot_tools as fplt
import fitting.high_level as fhl
from gpytorch.kernels import ScaleKernel as SK
from gpytorch.kernels import RBFKernel as RBF
import linear_operator
import fitting.models as models
import fitting.utils as futil
torch.set_default_dtype(torch.float64)
plt.close("auto")

%matplotlib inline

In [ ]:
res = AnalysisResult.fromFile("../results/data_control.pkl")

sample_manager = SampleManager()
sample_manager.loadSamplesFromDirectory("../datasets")

#res.results["CR0b_Data2018"].histograms["h_njet"] 
#res.results["CR0b_Data2018"].histograms["h_njet"] 

hists = res.getMergedHistograms(sample_manager)
sig_res = AnalysisResult.fromFile("../results/everything.pkl")
sig_res.results["signal_312_1500_600"].histograms["h_njet"] 
sighists = sig_res.getMergedHistograms(sample_manager)

In [ ]:
bkg_name="CR0b_Data2018"
#bkg_name="Skim_QCDInclusive2018"
complete_hist = hists["ratio_m14_vs_m24"]
mc = sighists["ratio_m14_vs_m24"]["Skim_QCDInclusive2018",...]
integral_sr = mc.sum()
integral_cr = complete_hist[bkg_name, ...].sum()
print(f"CR Integral = {integral_cr}")
print(f"SR Integral = {integral_sr}")
ratio = integral_sr.value / integral_cr.value
print(f"Ratio is  = {ratio}")
narrowed =  complete_hist[...,hist.loc(1000):hist.loc(3000),hist.loc(0.35):hist.loc(1.0)]
narrowed = narrowed[...,::hist.rebin(1),::hist.rebin(1)]
qcd_hist = narrowed[bkg_name,...]
sig_hist = sighists["ratio_m14_vs_m24"]["signal_312_1500_900",hist.loc(1000):hist.loc(3000),hist.loc(0.35):hist.loc(1)]

qcd_hist = narrowed[bkg_name,...]  * ratio
true_sum = qcd_hist.sum()
print(true_sum)


In [ ]:
sig_hist.sum()

In [ ]:
fig,ax= plt.subplots(1,2,figsize=(10,3))
_ = drawAs2DHist(ax[0], PlotObject.fromHist(qcd_hist))
_ = drawAs2DHist(ax[1], PlotObject.fromHist(sig_hist))

In [ ]:
import fitting.regression as fr
from fitting.windowing import EllipseWindow
from fitting.transformations import getNormalizationTransform

window = [(1300, 1750), (400, 700)]
rebin = 1
#mask_f = fr.ellipseMasker(torch.tensor([1400.0,0.6]), 200.0, 0.05)
mask_f = EllipseWindow([1420.0,0.58], [160.0,0.045])
#mask_f = fr.ellipseMasker(torch.tensor([2000.0,0.6]), 150.0,0.05)
#mask_f = fr.ellipseMasker(torch.tensor([1200.0,0.5]), 100.0,0.05)
#mask_f = fr.ellipseMasker(torch.tensor([1800.0,0.6]), 50.0,0.02)



#mask_f = fr.ellipseMasker(torch.tensor([1500.0,1400.0]), 200.0,150.0)
#mask_f = fr.ellipseMasker(torch.tensor([1450.0,0.59]), 150.0,0.05)
#mask_f=None

#mask_f = fr.ellipseMasker(torch.tensor([1200.0,0.6]), 200.0,0.05)
exclude_less=0.001
#mask_f = None
train_data = regression.makeRegressionData(qcd_hist[hist.rebin(rebin),hist.rebin(rebin)], mask_f, exclude_less=exclude_less)
test_data, m = regression.makeRegressionData(qcd_hist[hist.rebin(rebin),hist.rebin(rebin)], None, exclude_less=exclude_less, get_mask=True)
train_transform = getNormalizationTransform(train_data)
normalized_train_data = train_transform.transform(train_data)
normalized_test_data = train_transform.transform(test_data)

fig,ax = plt.subplots(2,2,figsize=(10,10),layout="tight")
fplt.simpleGrid(ax[0][0], train_data.E, train_data.X, train_data.Y)
mask = regression.getBlindedMask(test_data.X, test_data.Y, test_data.Y, test_data.V, mask_f)
squares = fplt.makeSquares(test_data.X[mask], test_data.E)
points = fplt.getPolyFromSquares(squares)
_=drawAs2DHist(ax[0][1], PlotObject.fromHist(sig_hist))
poly = matplotlib.patches.Polygon(points, edgecolor="red", fill=False)
ax[0][1].add_patch(poly)
signal_data = regression.makeRegressionData(sig_hist[hist.rebin(rebin),hist.rebin(rebin)])
signal_data = regression.DataValues(signal_data.X[m], signal_data.Y[m], signal_data.V[m], signal_data.E)
fplt.simpleGrid(ax[1][0], signal_data.E, signal_data.X, signal_data.Y)
fplt.simpleGrid(ax[1][1], normalized_train_data.E, normalized_train_data.X, normalized_train_data.V)

In [ ]:
from torchvision.models.densenet import DenseNet

class DenseNetFeatureExtractor(DenseNet):
    def forward(self, x):
        features = self.features(x)
        out = F.relu(features, inplace=True)
        out = F.avg_pool2d(out, kernel_size=self.avgpool_size).view(features.size(0), -1)
        return out

feature_extractor = DenseNetFeatureExtractor(block_config=(6, 6, 6), num_classes=2)
num_features = feature_extractor.classifier.in_features

In [ ]:
class ApproximateGPModel(gpytorch.models.ApproximateGP):
    def __init__(self, inducing_points):
        variational_distribution = gpytorch.variational.CholeskyVariationalDistribution(inducing_points.size(0))
        variational_strategy = gpytorch.variational.VariationalStrategy(
            self, inducing_points, variational_distribution, learn_inducing_locations=True
        )
        super().__init__(variational_strategy)
        self.mean_module = gpytorch.means.ConstantMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(gpytorch.kernels.RBFKernel())

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

data_dim = test_data.X.size(-1)

class LargeFeatureExtractor(torch.nn.Sequential):
    def __init__(self):
        super(LargeFeatureExtractor, self).__init__()
        self.add_module('linear1', torch.nn.Linear(data_dim, 32))
        self.add_module('relu1', torch.nn.ReLU())
        self.add_module('linear2', torch.nn.Linear(32, 128))
        self.add_module('relu2', torch.nn.ReLU())
        self.add_module('linearx', torch.nn.Linear(128, 64))
        self.add_module('relux', torch.nn.ReLU())
        self.add_module('linear3', torch.nn.Linear(64, 32))
        self.add_module('relu3', torch.nn.ReLU())
        self.add_module('linear4', torch.nn.Linear(32, 16))
        self.add_module('relu4', torch.nn.ReLU())
        self.add_module('linear5', torch.nn.Linear(16, 2))

feature_extractor = LargeFeatureExtractor()



In [ ]:
use_cuda = True 
from fitting.models import NNRBFKernel
from gpytorch.
import fitting.models as fm
if torch.cuda.is_available() and use_cuda:   
    train=normalized_train_data.toGpu()
else:
    train = normalized_train_data

class GPRegressionModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super(GPRegressionModel, self).__init__(train_x, train_y, likelihood)
        self.mean_module = gpytorch.means.ConstantMean()
        #self.covar_module = gpytorch.kernels.ScaleKernel(gpytorch.kernels.RBFKernel(ard_num_dims=2))
        self.covar_module = gpytorch.kernels.InducingPointKernel(
            gpytorch.kernels.ScaleKernel(
                #fm.GeneralRBF(ard_num_dims=2) 
                fm.GeneralRBF(ard_num_dims=2) 

                                        ),
            train_x[::].clone(),
            likelihood,
            
        )

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)


likelihood = gpytorch.likelihoods.FixedNoiseGaussianLikelihood(noise=train.V, learn_additional_noise=True)
model=GPRegressionModel(train.X,train.Y,likelihood)

if torch.cuda.is_available() and use_cuda:       
    model = model.cuda()
    likelihood = likelihood.cuda()
model,likelihood = regression.optimizeHyperparams(model,likelihood, 
                                                  train, 
                                                  iterations=400, lr=0.1)
if torch.cuda.is_available() and use_cuda:   
    model = model.cpu()
    likelihood = likelihood.cpu()


In [ ]:
#if mask_f:
  #  mask = regression.getBlindedMask(pred.X, pred.Y, test_data.Y, test_data.V, mask_f)
raw_pred_dist = regression.getPrediction(model, likelihood, normalized_test_data)
with  gpytorch.settings.cholesky_max_tries(30):
    pred_dist = futil.fixMVN(raw_pred_dist)
slope=train_transform.transform_y.slope
intercept = train_transform.transform_y.intercept
print(slope,intercept)

pred_dist = futil.affineTransformMVN(pred_dist,slope,  intercept)
#pred_dist = likelihood(pred_dist)
#pred_tr = train_transform.iTransform(regression.DataValues(normalized_test_data.X, pred_dist.mean, pred_dist.variance.detach(), normalized_test_data.E))
print(pred_dist.mean)
pred = regression.DataValues(test_data.X, pred_dist.mean, pred_dist.variance.detach(), test_data.E)

#print(all(pred_tr.Y == pred.Y))

if mask_f:
    mask = regression.getBlindedMask(pred.X, pred.Y, test_data.Y, test_data.V, mask_f)
    bpred_mean = pred.Y[mask]
    obs_mean = test_data.Y[mask]
    obs_var = test_data.V[mask]

    print(f"Total is {torch.sum(obs_mean)} -- {torch.sqrt(torch.sum(obs_mean))}")
    print(f"Approx Sigma is {torch.sum(signal_data.Y[mask])/torch.sqrt(torch.sum(obs_mean))}")
    chi2 = torch.sum((obs_mean - bpred_mean)**2 / obs_var) / torch.count_nonzero(mask)

    #print(f"Global Chi^2/bins = {}")

    chi2 = torch.sum((obs_mean - bpred_mean)**2 / obs_var) / torch.count_nonzero(mask)
    avg_pull = torch.sum(torch.abs((obs_mean - bpred_mean)) / torch.sqrt(obs_var)) / torch.count_nonzero(mask)
    print(f"Chi^2/bins = {chi2}")
    print(f"Avg Abs pull = {avg_pull}")
else:
    mask=None
if avg_pull < 5:    
    p = fhl.makeDiagnosticPlots(pred, test_data, train_data, qcd_hist, mask)

In [ ]:
def makeCovarPlot(target, points, model):
    return model(target, points)
x = makeCovarPlot(
    train_transform.transform_x.transformData(torch.tensor([[1800,0.65]])), train_transform.transform_x.transformData(pred.X), model.covar_module)
x = torch.squeeze(x.to_dense().detach())
fig,ax=plt.subplots()
fplt.simpleGrid(ax, pred.E, pred.X, x)

In [ ]:
params = list(model.named_parameters_and_constraints())
for name,param,constraint in params:
    if constraint:
        real_param = constraint.transform(param)
    else:
        real_param = param
    print(f"{name.replace('raw_',''):{max(len(x[0]) for x in params)+4}} {real_param.detach().numpy().round(3)}" )